<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/03_vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ベクトル検索 / セマンティック検索 (ニュース見出し)

このノートブックは **pytidb シリーズの第 3 回** です。ニュース記事の見出しを題材に、自然言語で意味的に近い記事を引き出す手順を学びます。

## 学習目標
- `EmbeddingFunction(model_name="tidbcloud_free/...")` で TiDB 組み込みの埋め込み関数を使う
- `VectorField(source_field="...")` で自動埋め込み (auto-embedding) 列を作る
- `table.search(QUERY)` で類似検索、`distance` / `similarity_score` を読み取る
- `distance_threshold` / `filter` と併用して絞り込みを行う

前提: `00_quickstart.ipynb` / `01_schema_and_types.ipynb` を実行済み。

この演習では、TiDB Cloud の無料プランで利用できる埋め込みモデル `tidbcloud_free/amazon/titan-embed-text-v2` を使います。そのためembeddingモデルのインストールや、APIキーの設定は不要です。


## 1. 依存パッケージのインストールとTiDB Cloudクラスタの作成


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-vector")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. 見出しテーブルを作成

ニュース記事の見出しを格納するテーブルを作ります。ここでは、`headline` 列に対して自動で埋め込み (`headline_vec`) を生成する構成にします。

`EmbeddingFunction("tidbcloud_free/amazon/titan-embed-text-v2")` はTiDB側で埋め込みを実行するため、pytidb側でのembeddingモデルのインストールやAPIキーの設定は不要です。

auto embeddingを実行する、`_embed.VectorField(source_field="headline")` の部分に注目してください。

In [ ]:
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.embeddings import EmbeddingFunction
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="news_demo",
    ensure_db=True,
)

_embed = EmbeddingFunction(
    model_name="tidbcloud_free/amazon/titan-embed-text-v2",
)


class Headline(TableModel):
    __tablename__ = "headlines"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    category: str = Field()
    headline: str = Field(sa_type=TEXT)
    # auto embedding. `headline` を埋め込み、`headline_vec` に格納する
    headline_vec: list[float] = _embed.VectorField(source_field="headline")


table = db.create_table(schema=Headline, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 3. サンプル見出しを投入

20件のフィクションの見出しをINSERTします。カテゴリは `tech` / `sports` / `food` / `science` の 4つです。


In [ ]:
HEADLINES = [
    ("tech",    "スタートアップが量子暗号を使った新メッセンジャーを発表"),
    ("tech",    "大手クラウドが東京にAI専用データセンターを新設"),
    ("tech",    "オープンソースの日本語LLMがGitHubで急上昇"),
    ("tech",    "自動運転トラックが都市間配送で商用試験を開始"),
    ("tech",    "スマホ向けARアプリが教育市場で利用を拡大"),
    ("sports",  "マラソン大会で国内新記録が誕生"),
    ("sports",  "プロ野球セ・リーグ、延長12回の末にサヨナラ勝ち"),
    ("sports",  "ワールドカップ予選、日本代表が欧州遠征で勝利"),
    ("sports",  "卓球の国際大会で高校生選手が初優勝"),
    ("sports",  "女子バスケットボール、東京チームが逆転で連勝"),
    ("food",    "地方発のクラフトビール醸造所が全国展開を開始"),
    ("food",    "新しい培養肉のバーガーが東京のレストランで試験提供"),
    ("food",    "家庭向けの自動調理家電、新モデルが30分で8品作る"),
    ("food",    "北海道のワイナリー、リースリングが国際コンクールで金賞"),
    ("food",    "冷凍技術が進化し、寿司ネタが自宅で解凍1分で食べ頃に"),
    ("science", "国内天文台が近傍銀河の暗黒物質分布を新たに解明"),
    ("science", "深海探査ロボットが新種の甲殻類を発見"),
    ("science", "mRNAを使った新しいワクチンの臨床試験が第3相に進む"),
    ("science", "ペロブスカイト太陽電池の屋外耐久性が大幅に向上"),
    ("science", "量子コンピュータで物質相転移のシミュレーションに成功"),
]

table.bulk_insert([
    Headline(id=i + 1, category=c, headline=h)
    for i, (c, h) in enumerate(HEADLINES)
])
print(f"投入完了: {table.rows()} 件")


## 4. ベクトル検索を実行

auto embedding列を使った検索では、`table.search(QUERY)` にテキストを渡すだけで、自動的にベクトルへ変換されて類似検索が走ります。
結果は `distance` (小さいほど近い) と `similarity_score = 1 - distance` を持ちます。


In [ ]:
QUERY = "宇宙や物理学の新しい発見について知りたい"

results = table.search(QUERY).limit(5).to_pydantic()

print(f"Query: {QUERY}\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r.hit.category}] {r.hit.headline}")
    print(f"    distance={r.distance:.4f}  similarity={r.similarity_score:.4f}")


## 5. フィルタと閾値を組み合わせる

メソッドチェーンで、カテゴリの絞り込みや、ある程度似ているものだけに限定することができます。


In [ ]:
# food カテゴリの中から意味的に近い上位 3 件
results = (
    table.search("自宅で簡単に新しい食体験ができる話題")
    .filter({"category": "food"})
    .limit(3)
    .to_pydantic()
)
print("[food カテゴリ内で意味検索]")
for r in results:
    print(f"  sim={r.similarity_score:.3f}  {r.hit.headline}")

# 閾値以下 (= 十分近い) のものだけ
print("\n[distance <= 0.6 に限定]")
results = (
    table.search("AIやデータセンターの動向")
    .distance_threshold(0.6)
    .limit(10)
    .to_pydantic()
)
for r in results:
    print(f"  d={r.distance:.3f}  [{r.hit.category}] {r.hit.headline}")


## 6. pytidbが行っているTiDB側の処理

pytidbのコードはTiDBのSQLを生成しています。今回の例でいうと、以下のような処理がTiDB側で行われています。

- サーバサイド埋め込み: `INSERT` 時に TiDB が `EMBED_TEXT()` 関数で `headline` を `headline_vec` に変換して保存
- 検索時も同じモデルで質問文が埋め込まれ、`VEC_COSINE_DISTANCE` でスコアを計算
- デフォルトの距離メトリクスは COSINE、HNSW インデックスが自動で貼られる




## 次のステップ

- `04_fulltext_search.ipynb` : キーワード一致ベースの全文検索
- `05_hybrid_search.ipynb` : ベクトル + 全文のハイブリッド

## チャレンジ

- クエリを `"おいしい料理の最新ニュース"` に変えて結果がどう変わるか
- `distance_threshold(0.3)` にして、ヒットが 0 件になる境界を探す
- 独自の見出しを追加 (`table.insert(...)`) し、再検索で上位に来るか確認
